In [ ]:
import numpy as np

# ===== RADAR PARAMETERS =====
c = 3e8
fc = 77e9
R_max = 50
range_resolution = 1

B = c / (2 * range_resolution)
T_chirp = 4e-6
slope = B / T_chirp
t_max = 2 * R_max / c

Nr = 1024
Nd = 1024

t = np.linspace(0, T_chirp, Nr)
dt = t[1] - t[0]

# ===== TARGETS =====
targets = [
    {"R": 35.0, "az": np.deg2rad(30),  "el": np.deg2rad(45), "v": 100.0},
    {"R": 40.0, "az": np.deg2rad(330), "el": np.deg2rad(15), "v": -80.0},
    {"R": 30.0, "az": np.deg2rad(130), "el": np.deg2rad(75), "v": -1.0},
    {"R": 10.0, "az": np.deg2rad(30),  "el": np.deg2rad(45), "v": 100.0},
]

N_rx = 4

# ===== SIGNAL GENERATION =====
def generate_rx_single(t, slope, tau, d, lam, az, el):
    t_delayed = t - tau
    base = np.exp(1j * 2 * np.pi * (fc * t_delayed + (slope * t_delayed**2) / 2))

    def noise():
        return 0.02 * (np.random.randn(len(t)) + 1j * np.random.randn(len(t)))

    phi_x = (2 * np.pi * d / lam) * np.cos(el) * np.cos(az)
    phi_y = (2 * np.pi * d / lam) * np.cos(el) * np.sin(az)
    phi_z = (2 * np.pi * d / lam) * np.sin(el)

    Rx1 = base + noise()
    Rx2 = base * np.exp(1j * phi_x) + noise()
    Rx3 = base * np.exp(1j * phi_y) + noise()
    Rx4 = base * np.exp(1j * phi_z) + noise()

    return Rx1, Rx2, Rx3, Rx4


def generate_combined_rx(targets, t, slope, d, lam):
    Rx1_sum = np.zeros(len(t), dtype=complex)
    Rx2_sum = np.zeros(len(t), dtype=complex)
    Rx3_sum = np.zeros(len(t), dtype=complex)
    Rx4_sum = np.zeros(len(t), dtype=complex)

    for tgt in targets:
        tau = 2 * tgt["R"] / c
        if tau <= t_max:
            r1, r2, r3, r4 = generate_rx_single(
                t, slope, tau, d, lam, tgt["az"], tgt["el"]
            )
            Rx1_sum += r1
            Rx2_sum += r2
            Rx3_sum += r3
            Rx4_sum += r4

    return Rx1_sum, Rx2_sum, Rx3_sum, Rx4_sum


print("targets and RX generation ready.")

In [ ]:
from scipy.signal import find_peaks

# ===== BUILD RD CUBE =====
def build_rd_cube(targets, Nd, t, slope, d, lam):
    raw = np.zeros((N_rx, Nd, Nr), dtype=complex)

    for chirp_idx in range(Nd):
        targets_at_chirp = [
            {**tgt, "R": tgt["R"] + tgt["v"] * chirp_idx * T_chirp}
            for tgt in targets
        ]
        Rx_all = generate_combined_rx(targets_at_chirp, t, slope, d, lam)

        for m, Rxm in enumerate(Rx_all):
            beat = Tx * np.conj(Rxm)
            beat *= np.hamming(Nr)
            raw[m, chirp_idx, :] = beat

    range_fft = np.fft.fft(raw, axis=2)

    global range_fft_matrix
    range_fft_matrix = range_fft[0]   # for plotting

    doppler_window = np.hamming(Nd)
    range_fft_win = range_fft * doppler_window[np.newaxis, :, np.newaxis]
    rd_cube = np.fft.fft(range_fft_win, axis=1)
    rd_cube = np.fft.fftshift(rd_cube, axes=1)
    return rd_cube

# ===== RD DETECTION =====
def detect_peaks_from_cube(rd_cube, range_axis_full, velocity_axis_full):
    rd_mag = np.abs(rd_cube[0])

    pos_mask       = range_axis_full > 0
    range_axis_pos = range_axis_full[pos_mask]
    rd_pos         = rd_mag[:, pos_mask]

    noise_floor = np.mean(rd_pos) + 6 * np.std(rd_pos)

    raw = []
    for r_idx in range(rd_pos.shape[1]):
        col = rd_pos[:, r_idx]

        peaks, props = find_peaks(col, height=noise_floor)

        if len(peaks) > 0:
            # keep ONLY strongest peak in this range bin
            best = peaks[np.argmax(props["peak_heights"])]

            raw.append((range_axis_pos[r_idx],
                        velocity_axis_full[best],
                        col[best], r_idx, best))

    raw.sort(key=lambda x: -x[2])

    final = []
    for det in raw:
        R, V = det[0], det[1]
        if not any(abs(R - f[0]) < 3.0 and abs(V - f[1]) < 2.0 for f in final):
            final.append(det)

    return final, range_axis_pos

# ===== SINGLE TARGET RD =====
def build_rd_cube_single_target(tgt, Nd, t, slope, d, lam):
    raw = np.zeros((N_rx, Nd, Nr), dtype=complex)
    for chirp_idx in range(Nd):
        tgt_now = {**tgt, "R": tgt["R"] + tgt["v"] * chirp_idx * T_chirp}
        rxs = generate_combined_rx([tgt_now], t, slope, d, lam)
        for m, rxm in enumerate(rxs):
            beat = Tx * np.conj(rxm)
            beat *= np.hamming(Nr)
            raw[m, chirp_idx, :] = beat
    range_fft = np.fft.fft(raw, axis=2)
    doppler_win = np.hamming(Nd)
    rd = np.fft.fft(
        range_fft * doppler_win[np.newaxis, :, np.newaxis], axis=1)
    rd = np.fft.fftshift(rd, axes=1)
    return rd


def direct_phase_doa(rd_cube_single, range_bin, doppler_bin, d, lam):
    v = [rd_cube_single[m, doppler_bin, range_bin] for m in range(N_rx)]

    dphi_x = -np.angle(np.conj(v[0]) * v[1])
    dphi_y = -np.angle(np.conj(v[0]) * v[2])
    dphi_z = -np.angle(np.conj(v[0]) * v[3])

    k = 2 * np.pi * d / lam

    sin_el = np.clip(dphi_z / k, -1, 1)
    el_est = float(np.arcsin(sin_el))

    az_est = float(np.arctan2(dphi_y, dphi_x) % (2 * np.pi))

    return az_est, el_est

# ===== MAIN =====
lam = c / fc
d   = lam / 2

Rx1_c1, Rx2_c1, Rx3_c1, Rx4_c1 = generate_combined_rx(targets, t, slope, d, lam)

range_axis_full    = (np.fft.fftfreq(Nr, d=dt) * c) / (2 * slope)
velocity_axis_full = np.fft.fftshift(np.fft.fftfreq(Nd, d=T_chirp)) * lam / 2

rd_cube = build_rd_cube(targets, Nd, t, slope, d, lam)

rd_map = rd_cube[0]
velocity_axis = velocity_axis_full

raw_detections, range_axis_pos = detect_peaks_from_cube(
    rd_cube, range_axis_full, velocity_axis_full
)

print(f"RD detections (before DOA): {len(raw_detections)}")

rd_cubes_per_target = [
    build_rd_cube_single_target(tgt, Nd, t, slope, d, lam)
    for tgt in targets
]

final_results = []

for det in raw_detections:
    R_est, V_est, _, _, _ = det
    range_bin   = np.argmin(np.abs(range_axis_full - R_est))
    doppler_bin = np.argmin(np.abs(velocity_axis_full - V_est))

        # Just use ONE cube (not per target)
    az_est, el_est = direct_phase_doa(
        rd_cube, range_bin, doppler_bin, d, lam)

    final_results.append({
        "R_est":  R_est,
        "V_est":  V_est,
        "az_est": az_est,
        "el_est": el_est,
    })

# ===== DEDUP =====
def is_same(a, b):
    az_diff = abs(np.rad2deg(a["az_est"]) - np.rad2deg(b["az_est"]))
    az_diff = min(az_diff, 360 - az_diff)
    return (abs(a["R_est"] - b["R_est"]) < 1.5 and
            abs(a["V_est"] - b["V_est"]) < 1.5 and
            az_diff < 5.0 and
            abs(np.rad2deg(a["el_est"] - b["el_est"])) < 3.0)

deduped = []
for r in final_results:
    if not any(is_same(r, f) for f in deduped):
        deduped.append(r)

results = []
for r in deduped:
    results.append({
        "R_est": r["R_est"],
        "theta_est": r["az_est"],
        "phi_est": r["el_est"],
        "v_radial": r["V_est"]
    })        

# ===== OUTPUT =====
print(f"\nDetected {len(deduped)} target(s)")

gt_used = []
for i, r in enumerate(deduped):
    best_idx, best_dist = None, np.inf
    for j, gt in enumerate(targets):
        if j in gt_used:
            continue
        az_diff = abs(np.rad2deg(r["az_est"]) - np.rad2deg(gt["az"]))
        az_diff = min(az_diff, 360 - az_diff)
        if az_diff < best_dist:
            best_dist, best_idx = az_diff, j
    if best_idx is not None:
        gt_used.append(best_idx)
    gt = targets[best_idx] if best_idx is not None else {}

    print(f"\n--- Target O{i+1} ---")
    print(f"Distance  : {round(r['R_est'], 2)} m")
    print(f"Velocity  : {round(r['V_est'], 2)} m/s")
    print(f"Azimuth   : {round(np.rad2deg(r['az_est']), 2)} deg")
    print(f"Elevation : {round(np.rad2deg(r['el_est']), 2)} deg")
    if gt:
        print(f"  [actual] R={round(gt['R'], 2)}m | v={round(gt['v'], 2)}m/s | "
              f"az={round(np.rad2deg(gt['az']),1)}° | "
              f"el={round(np.rad2deg(gt['el']),1)}°")

In [ ]:
#plot code

import numpy as np
import matplotlib.pyplot as plt

def wrap_to_360(angles_deg):
    return (angles_deg + 360) % 360

def wrap_to_180(angle_deg):
    return (angle_deg + 180) % 360 - 180

# ================================
# 🟢 1. SIGNAL LEVEL
# ================================
def plot_signals_per_target(targets, t, slope, d, lam):

    plt.figure(figsize=(12, 4*len(targets)))

    for i, tgt in enumerate(targets):

        # ===== SINGLE TARGET RX =====
        tau = 2 * tgt["R"] / c

        Rx1, _, _, _ = generate_rx_single(
            t, slope, tau, d, lam, tgt["az"], tgt["el"]
        )

        # ===== BEAT =====
        beat = Tx * np.conj(Rx1)

        # ===== PLOT =====
        plt.subplot(len(targets), 1, i+1)

        plt.plot(t, np.real(Tx), label="Tx", alpha=0.7)
        plt.plot(t, np.real(Rx1), label=f"Rx (Target O{i+1})", alpha=0.7)
        plt.plot(t, np.real(beat), label="Beat", alpha=0.7)

        plt.title(
            f"Target O{i+1} | R={tgt['R']}m | Az={round(np.rad2deg(tgt['az']),1)}°"
        )
        plt.xlabel("Time")
        plt.ylabel("Amplitude")
        plt.legend()
        plt.grid()

    plt.tight_layout()
    plt.show()

plot_signals_per_target(targets, t, slope, d, lam)



# ================================
# 🟢 2. RANGE FFT (FINAL VALUES)
# ================================
range_profile = np.mean(np.abs(range_fft_matrix), axis=0)
range_axis_full = (np.fft.fftfreq(Nr, d=dt) * c) / (2 * slope)

pos_mask = range_axis_full > 0

plt.figure(figsize=(10,4))
plt.plot(range_axis_full[pos_mask], range_profile[pos_mask])

# ✅ USE FINAL RESULTS (same as 2D/3D)
for i, r in enumerate(results):
    R = r["R_est"]
    az = wrap_to_360(np.rad2deg(r["theta_est"]))

    plt.axvline(R, color='red', linestyle='--')

    plt.text(
        R,
        np.max(range_profile)*0.75,
        f"O{i+1}\nR={round(R,2)}m",
        color='red',
        ha='center',
        bbox=dict(facecolor='white', alpha=0.7)
    )

plt.title("Range FFT (Range Profile)")
plt.xlabel("Range (m)")
plt.ylabel("Magnitude")
plt.grid()
plt.xlim(0, R_max)
plt.show()


# ================================
# 🟢 3. RANGE-DOPPLER MAP
def plot_range_doppler_map_full():
    rd_magnitude = np.abs(rd_map)
    rd_plot = rd_magnitude[:, range_axis_full > 0]

    plt.figure(figsize=(12,6))
    plt.imshow(
        20*np.log10(rd_plot + 1e-10),
        aspect='auto',
        extent=[range_axis_pos[0], range_axis_pos[-1],
                velocity_axis[0], velocity_axis[-1]],
        origin='lower',
        cmap='jet'
    )

    plt.colorbar(label='Magnitude (dB)')
    plt.xlabel('Range (m)')
    plt.ylabel('Velocity (m/s)')
    plt.title('Range-Doppler Map')
    plt.xlim(0, R_max)

    # ✅ USE SAME FINAL VALUES AS 2D/3D
    for i, r in enumerate(results):
        R  = r["R_est"]
        V  = r["v_radial"]
        az = wrap_to_360(np.rad2deg(r["theta_est"]))
        el = np.rad2deg(r["phi_est"])

        # marker
        plt.plot(R, V, 'wo', markersize=6)

        # label (FULL INFO)
        plt.text(
            R + 0.5,
            V + 0.3,
            f"O{i+1}\nR={round(R,2)}m\nV={round(V,2)}m/s",
            color='white',
            bbox=dict(facecolor='black', alpha=0.6)
        )

    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
plot_range_doppler_map_full()


# ================================
# 🟢 4. ANGLE SPECTRUM (unchanged)
# ================================
def plot_angle_spectrum_fixed(r_idx, d_idx):

    # snapshot (correct)
    X = np.array([
        rd_cube[0, d_idx, r_idx],
        rd_cube[1, d_idx, r_idx],
        rd_cube[2, d_idx, r_idx],
        rd_cube[3, d_idx, r_idx],
    ])

    angles = np.linspace(-90, 90, 361)
    response = []

    for ang in angles:
        theta = np.deg2rad(ang)

        # ✅ FIXED steering vector (MATCH DOA SIGN)
        a = np.array([
            1,
            np.exp(-1j * 2*np.pi*d*np.cos(theta)/lam),  # Rx2 (x)
            np.exp(-1j * 2*np.pi*d*np.sin(theta)/lam),  # Rx3 (y)
            1
        ])

        response.append(np.abs(np.conj(a) @ X))

    response = np.array(response)

    plt.figure()
    plt.plot(angles, 20*np.log10(response + 1e-6))
    plt.title("Angle Spectrum (CORRECTED)")
    plt.xlabel("Angle (deg)")
    plt.ylabel("Power (dB)")
    plt.grid()
    plt.show()


# ================================
# 🔴 5. MUSIC (FIXED PROPERLY)
# ================================
def plot_music_fixed(r_idx, d_idx):

    # multi-snapshot (correct)
    X = np.array([
        rd_cube[0, :, r_idx],
        rd_cube[1, :, r_idx],
        rd_cube[2, :, r_idx],
        rd_cube[3, :, r_idx],
    ])

    Rxx = X @ X.conj().T / X.shape[1]

    eigvals, eigvecs = np.linalg.eigh(Rxx)
    idx = np.argsort(eigvals)

    num_targets = 1
    noise_space = eigvecs[:, idx[:-num_targets]]

    angles = np.linspace(-90, 90, 361)
    spectrum = []

    for ang in angles:
        theta = np.deg2rad(ang)

        # ✅ FIXED steering vector (MATCH DOA SIGN)
        a = np.array([
            1,
            np.exp(-1j * 2*np.pi*d*np.cos(theta)/lam),
            np.exp(-1j * 2*np.pi*d*np.sin(theta)/lam),
            1
        ])

        val = 1.0 / (np.abs(a.conj().T @ noise_space @ noise_space.conj().T @ a) + 1e-12)
        spectrum.append(val)

    spectrum = np.array(spectrum)

    plt.figure()
    plt.plot(angles, 10*np.log10(spectrum + 1e-6))
    plt.title("MUSIC Spectrum (CORRECTED)")
    plt.xlabel("Angle (deg)")
    plt.ylabel("Power (dB)")
    plt.grid()
    plt.show()

def plot_az_el_combined(r_idx, d_idx, target_label):

    # ===== SNAPSHOTS =====
    X_single = np.array([
        rd_cube[0, d_idx, r_idx],
        rd_cube[1, d_idx, r_idx],
        rd_cube[2, d_idx, r_idx],
        rd_cube[3, d_idx, r_idx],
    ])

    X_multi = np.array([
        rd_cube[0, :, r_idx],
        rd_cube[1, :, r_idx],
        rd_cube[2, :, r_idx],
        rd_cube[3, :, r_idx],
    ])

    # ===== MUSIC PREP =====
    Rxx = X_multi @ X_multi.conj().T / X_multi.shape[1]
    eigvals, eigvecs = np.linalg.eig(Rxx)
    idx = np.argsort(eigvals.real)
    # FIX: cap signal subspace so noise subspace always has ≥1 dimension
    num_antennas = X_multi.shape[0]
    num_targets = min(len(results), num_antennas - 1)  # ← THIS LINE CHANGED
    noise_space = eigvecs[:, idx[:-num_targets]]

    # ===== ESTIMATED VALUES =====
    idx_num  = int(target_label[-1]) - 1
    az_est   = results[idx_num]["theta_est"]
    el_est   = results[idx_num]["phi_est"]
    az_est_d = wrap_to_360(np.rad2deg(az_est))
    el_est_d = np.rad2deg(el_est)

    k      = 2 * np.pi * d / lam
    cos_el = np.cos(el_est)

    # ===== EXTRACT MEASURED PHASE FROM DATA =====
    dphi_x = np.angle(np.conj(X_single[1]) * X_single[0])
    dphi_y = np.angle(np.conj(X_single[2]) * X_single[0])
    dphi_z = np.angle(np.conj(X_single[3]) * X_single[0])

    # ===== AZIMUTH SCAN =====
    az_angles = np.linspace(0, 360, 721)
    az_spec  = []
    az_music = []

    for ang in az_angles:
        theta = np.deg2rad(ang)

        # Expected phases from signal model in main code:
        # Rx2: exp(j * k * cos(el) * cos(az))
        # Rx3: exp(j * k * cos(el) * sin(az))
        expected_x = k * cos_el * np.cos(theta)
        expected_y = k * cos_el * np.sin(theta)

        # Phase match score: 1 when measured == expected
        match_x = np.cos(dphi_x - expected_x)
        match_y = np.cos(dphi_y - expected_y)

        # Both must agree for a true peak
        response = (match_x + 1) * (match_y + 1) / 4.0
        az_spec.append(max(response, 1e-10))

        # MUSIC: match sign convention of signal generation
        # phi_x =  k * cos(el) * cos(az)  → Rx2 = base * exp(+j*phi_x)
        # phi_y =  k * cos(el) * sin(az)  → Rx3 = base * exp(+j*phi_y)
        # phi_z =  k * sin(el)            → Rx4 = base * exp(+j*phi_z)
        a = np.array([
            1,
            np.exp(+1j * k * cos_el * np.cos(theta)),
            np.exp(+1j * k * cos_el * np.sin(theta)),
            np.exp(+1j * k * np.sin(el_est)),
        ])
        val = 1.0 / (np.abs(a @ noise_space @ noise_space.conj().T @ a.conj()) + 1e-12)
        az_music.append(val)

    az_spec  = np.array(az_spec)
    az_music = np.array(az_music)

    # ===== ELEVATION — Rx1 vs Rx4 phase scan =====
    el_angles = np.linspace(-90, 90, 361)
    el_spec   = []

    for ang in el_angles:
        ang_r = np.deg2rad(ang)
        # Rx4 = base * exp(+j * k * sin(el))
        expected_z = k * np.sin(ang_r)
        match_z = (np.cos(dphi_z - expected_z) + 1) / 2.0
        el_spec.append(max(match_z, 1e-10))

    el_spec = np.array(el_spec)

    # ===== PLOT — original layout =====
    fig = plt.figure(figsize=(12, 8))

    ax1 = plt.subplot2grid((2, 2), (0, 0))
    ax2 = plt.subplot2grid((2, 2), (0, 1))
    ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)

    # AZ Spectrum
    ax1.plot(az_angles, 20 * np.log10(az_spec))
    ax1.axvline(az_est_d, color='r', linestyle='--', label=f"Est={round(az_est_d,1)}°")
    ax1.set_title(f"{target_label} - Azimuth Spectrum")
    ax1.set_xlabel("Azimuth (deg)")
    ax1.set_ylabel("Power (dB)")
    ax1.set_xlim(0, 360)
    ax1.legend()
    ax1.grid()

    # AZ MUSIC
    ax2.plot(az_angles, 10 * np.log10(az_music + 1e-6))
    ax2.axvline(az_est_d, color='r', linestyle='--', label=f"Est={round(az_est_d,1)}°")
    ax2.set_title(f"{target_label} - Azimuth MUSIC")
    ax2.set_xlabel("Azimuth (deg)")
    ax2.set_ylabel("Power (dB)")
    ax2.set_xlim(0, 360)
    ax2.legend()
    ax2.grid()

    # ELEVATION
    ax3.plot(el_angles, 20 * np.log10(el_spec))
    ax3.axvline(el_est_d, color='r', linestyle='--', label=f"Est={round(el_est_d,1)}°")
    ax3.set_title(f"{target_label} - Elevation Spectrum (Rx1 vs Rx4)")
    ax3.set_xlabel("Elevation (deg)")
    ax3.set_ylabel("Power (dB)")
    ax3.set_xlim(-90, 90)
    ax3.legend()
    ax3.grid()

    plt.tight_layout()
    plt.show()


# ================================
# 🔥 CORRECT LOOP
# ================================
for i, det in enumerate(raw_detections):
    _, _, _, r_idx, d_idx = det

    plot_az_el_combined(r_idx, d_idx, f"Target O{i+1}")
# ================================
# 🟢 6. 2D RADAR (YOUR STYLE)
# ================================
def plot_all_2D(results):
    limit = R_max + 10
    offset = 4
    arrow_scale = 6

    plt.figure(figsize=(9, 9))
    plt.scatter(0, 0, color='red', s=100, label='Radar')
    plt.scatter([], [], color='blue', s=100, label='Target')

    ax = plt.gca()

    for i, res in enumerate(results):
        R_est     = res["R_est"]
        theta_est = res["theta_est"]
        v_radial  = res["v_radial"]
        label     = f"O{i+1}"

        x = R_est * np.cos(theta_est)
        y = R_est * np.sin(theta_est)

        ux = np.cos(theta_est)
        uy = np.sin(theta_est)

        if v_radial < 0:
            dvx, dvy = -ux * arrow_scale, -uy * arrow_scale
            v_color = 'black'
        elif v_radial > 0:
            dvx, dvy =  ux * arrow_scale,  uy * arrow_scale
            v_color = 'orange'
        else:
            dvx, dvy = 0, 0
            v_color = 'gray'

        # Distance line
        plt.plot([0, x], [0, y], linewidth=2, color='green')

        # Distance label
        plt.text(x/2, y/2, f"{round(R_est,2)}m", fontsize=9, color='black')

        # Target circle label
        plt.text(x, y, label, fontsize=10, color='white',
                 ha='center', va='center',
                 bbox=dict(facecolor='blue', edgecolor='none',
                           boxstyle='circle,pad=0.4'))

        # Velocity arrow
        if dvx != 0 or dvy != 0:
            plt.annotate('',
                xy=(x + dvx, y + dvy),
                xytext=(x, y),
                arrowprops=dict(
                    arrowstyle='-|>',
                    color=v_color,
                    lw=1.2,
                    mutation_scale=20
                )
            )
            label_offset = -4 if v_radial > 0 else 4
            plt.text(
                x, y + label_offset,
                f"{round(abs(v_radial), 2)} m/s",
                fontsize=9, color='white', ha='center', va='center',
                bbox=dict(facecolor='black', edgecolor=v_color, boxstyle='round,pad=0.3')
                )
        else:
            plt.text(x + 1.5, y + 1.5, "0.0 m/s", fontsize=9, color='white',
                     bbox=dict(facecolor='black', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.xlim(-limit, limit)
    plt.ylim(-limit, limit)
    ax.set_aspect('equal', adjustable='box')
    plt.title('Radar Detection - Multiple Targets')
    plt.grid(True, linestyle='--', alpha=0.6)

    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

    plt.text(limit, offset, '+X', ha='center')
    plt.text(-limit, offset, '-X', ha='center')
    plt.text(offset, limit, '+Y', va='center')
    plt.text(offset, -limit, '-Y', va='center')

    plt.legend()
    plt.show()


# ================================
# 🟢 7. 3D RADAR
# ================================
def plot_all_3D(results):
    arrow_scale = 8

    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot([-R_max, R_max], [0, 0], [0, 0], color='gray', linewidth=0.8, linestyle='--')
    ax.plot([0, 0], [-R_max, R_max], [0, 0], color='gray', linewidth=0.8, linestyle='--')
    ax.plot([0, 0], [0, 0], [0, R_max], color='gray', linewidth=0.8, linestyle='--')
    ax.text(R_max, 0, 0,  '+X', color='black', fontsize=9)
    ax.text(-R_max, 0, 0, '-X', color='black', fontsize=9)
    ax.text(0, R_max, 0,  '+Y', color='black', fontsize=9)
    ax.text(0, -R_max, 0, '-Y', color='black', fontsize=9)
    ax.text(0, 0, R_max,  '+Z', color='black', fontsize=9)

    ax.scatter(0, 0, 0, color='red', s=100, label='Radar')

    for i, res in enumerate(results):
        R_est     = res["R_est"]
        theta_est = res["theta_est"]
        phi_est   = res["phi_est"]
        v_radial  = res["v_radial"]
        label     = f"O{i+1}"

        x = R_est * np.cos(theta_est) * np.cos(phi_est)
        y = R_est * np.sin(theta_est) * np.cos(phi_est)
        z = R_est * np.sin(phi_est)

        ux = np.cos(theta_est) * np.cos(phi_est)
        uy = np.sin(theta_est) * np.cos(phi_est)
        uz = np.sin(phi_est)

        if v_radial < 0:
            dvx, dvy, dvz = -ux * arrow_scale, -uy * arrow_scale, -uz * arrow_scale
            v_color = 'black'
        elif v_radial > 0:
            dvx, dvy, dvz =  ux * arrow_scale,  uy * arrow_scale,  uz * arrow_scale
            v_color = 'orange'
        else:
            dvx, dvy, dvz = 0, 0, 0
            v_color = 'gray'

        # Distance line + label
        ax.plot([0, x], [0, y], [0, z], color='green', linewidth=2)
        ax.text(x/2, y/2, z/2, f"{round(R_est,2)}m", color='black')

        # Target circle label
        ax.text(x, y, z, label, fontsize=6, color='white',
                ha='center', va='center',
                bbox=dict(facecolor='blue', edgecolor='none',
                          boxstyle='circle,pad=0.4'))

        # replace the label part in plot_all_3D:
        if dvx != 0 or dvy != 0 or dvz != 0:
            ax.quiver(x, y, z, dvx, dvy, dvz,
                    color=v_color, linewidth=0.8, arrow_length_ratio=0.25)
            
            # perpendicular offset in XY plane to avoid overlapping arrow
            perp_x = -uy / (np.hypot(ux, uy) + 1e-9)
            perp_y =  ux / (np.hypot(ux, uy) + 1e-9)
            
            ax.text(
                x + 3, y + 3, z + 3,
                f"{round(abs(v_radial), 2)} m/s",
                color='white', fontsize=9,
                bbox=dict(facecolor='black', edgecolor=v_color, boxstyle='round,pad=0.3')
                )
        else:
            ax.text(x + 0.5, y + 0.5, z + 0.5, "0.0 m/s",
                    color='white', fontsize=9,
                    bbox=dict(facecolor='black', edgecolor='gray', boxstyle='round,pad=0.3'))

    ax.set_xlim(-R_max, R_max)
    ax.set_ylim(-R_max, R_max)
    ax.set_zlim(0, R_max)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('3D Radar Detection - Multiple Targets')
    ax.legend()
    plt.show()


plot_all_2D(results)
plot_all_3D(results)